### I am using this notebook for Iris embeddings → store in ChromaDB 

In [16]:
import chromadb
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
import numpy as np

In [17]:
# --- Load Iris ---
iris = load_iris()
X, y = iris.data, iris.target
target_names = iris.target_names
feature_names = iris.feature_names

In [18]:
# --- Create embeddings ---
# Iris is tabular (4 numeric features), so we use the standardized
# feature vector itself as the "embedding" (common approach for tabular data
# when there's no pretrained encoder available).
scaler = StandardScaler()
embeddings = scaler.fit_transform(X)  # shape: (150, 4)

In [19]:
# --- Set up ChromaDB (persistent local store) ---
client = chromadb.PersistentClient(path="./chroma_db")

In [20]:
# Create (or reset) collection
try:
    client.delete_collection("iris_embeddings")
except Exception:
    pass

collection = client.create_collection(
    name="iris_embeddings",
    metadata={"description": "Iris dataset stored as embeddings (standardized features)"}
)

In [21]:
# --- Prepare data for insertion ---
ids = [f"iris_{i}" for i in range(len(X))]
documents = [
    f"Sample {i}: sepal_length={row[0]}, sepal_width={row[1]}, "
    f"petal_length={row[2]}, petal_width={row[3]}, species={target_names[y[i]]}"
    for i, row in enumerate(X)
]
metadatas = [
    {
        "species": target_names[y[i]],
        "sepal_length": float(X[i][0]),
        "sepal_width": float(X[i][1]),
        "petal_length": float(X[i][2]),
        "petal_width": float(X[i][3]),
    }
    for i in range(len(X))
]

In [22]:
# --- Insert into ChromaDB ---
collection.add(
    ids=ids,
    embeddings=embeddings.tolist(),
    documents=documents,
    metadatas=metadatas,
)

print(f"Inserted {collection.count()} Iris embeddings into ChromaDB.")
print(f"Collection: {collection.name}")

Inserted 150 Iris embeddings into ChromaDB.
Collection: iris_embeddings


In [23]:
# --- Quick sanity check: query nearest neighbors for sample 0 ---
query_result = collection.query(
    query_embeddings=[embeddings[0].tolist()],
    n_results=5
)

print("\n=== Nearest neighbors to sample 0 (self-match expected first) ===")
for i, (doc_id, dist, meta) in enumerate(zip(
    query_result["ids"][0], query_result["distances"][0], query_result["metadatas"][0]
)):
    print(f"{i+1}. id={doc_id} | species={meta['species']} | distance={dist:.4f}")


=== Nearest neighbors to sample 0 (self-match expected first) ===
1. id=iris_0 | species=setosa | distance=0.0000
2. id=iris_17 | species=setosa | distance=0.0173
3. id=iris_27 | species=setosa | distance=0.0179
4. id=iris_40 | species=setosa | distance=0.0352
5. id=iris_39 | species=setosa | distance=0.0562
